# Phase 5 - SHAP Explainability

SHAP describes the saved Phase 4 model; it does not select or change the model. For this selected Logistic Regression model, contributions are in Default=1 log-odds, not probability points.

## 1. Load final model

In [ ]:
from pathlib import Path
import json
import sys

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import shap

CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent, Path.cwd() / "loan-default-prediction", Path.cwd().parent / "loan-default-prediction"]
ROOT = next((candidate for candidate in CANDIDATE_ROOTS if (candidate / "models/best_model.joblib").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Could not locate models/best_model.joblib")
sys.path.insert(0, str(ROOT))
from scripts.optimize_model import load_original_split
from src.explainability.shap_explainer import LoanSHAPExplainer

explainability = LoanSHAPExplainer()
metadata = json.loads((ROOT / "models/best_model_metadata.json").read_text(encoding="utf-8"))
print("Model:", metadata["model_name"], "| explainer:", explainability.explainer_type)
print("SHAP output units:", explainability.output_units)

## 2. Load validation data

In [ ]:
validation = load_original_split("validation").drop(columns="Default")
print(f"Validation rows available: {len(validation):,}")

## 3. Reuse Phase 2 preprocessing

In [ ]:
preprocessor = explainability.preprocessor
feature_names = explainability.get_feature_names()
print(f"Fitted transformer: {type(preprocessor).__name__}; transformed columns: {len(feature_names)}")
display(pd.DataFrame({"feature": feature_names}).head(15))

## 4. Create SHAP explainer

## 5. Sample validation data

In [ ]:
sample_size = min(5000, len(validation))
sample = validation.sample(n=sample_size, random_state=42).reset_index(drop=True)
print(f"Validation records selected for explanation: {len(sample):,}")

In [ ]:
background = validation.sample(
    n=min(explainability.background_size, len(validation)), random_state=42
)
shap_explainer = explainability.create_explainer(background)
print(f"Validation background rows: {len(background):,}")

## 6. Calculate SHAP values

In [ ]:
shap_values = explainability.calculate_shap_values(sample, background_data=background)
print("SHAP shape:", shap_values.values.shape)
print("Contributions are for Default=1 in:", explainability.output_units)

## 7. Global feature importance

In [ ]:
importance = explainability.get_global_importance(shap_values)
display(importance.head(20))

## 8. SHAP summary bar plot

In [ ]:
shap.plots.bar(shap_values, max_display=20)

## 9. SHAP beeswarm plot

In [ ]:
shap.plots.beeswarm(shap_values, max_display=20)

## 10. Individual loan explanation

In [ ]:
local_loan = sample.iloc[[0]]
local_result = explainability.explain_prediction(local_loan)
print(json.dumps(local_result, indent=2))

## 11. Local SHAP waterfall plot

In [ ]:
local_values = explainability.calculate_shap_values(local_loan, background_data=background)
shap.plots.waterfall(local_values[0], max_display=15)

## 12. Key observations

In [ ]:
print("Highest mean absolute contributions in this validation sample:")
for row in importance.head(10).itertuples(index=False):
    print(f"{row.rank}. {row.feature}: {row.mean_abs_shap:.6f}")
print()
print("SHAP describes model behavior, not causality or the effect of changing a borrower's real-world outcome.")